## 3.3 Final Validated Political Video Set

**Author:** Kristina Aleksandrovna Pedersen

This code shows how we generated the validated political video set and established the final strategy categories for each video. This final dataset of validated strategies was then used as the source for the final sample of political videos provided to the respondents in the simulator.

Note that the RAs were initiated in three seperate coding rounds (for the same sets of videos). Rules for strategy determination are detailed further below.


In [ ]:
import json
import pandas as pd
import os
import numpy as np
import polars as pl
from sklearn.metrics import cohen_kappa_score
from IPython.display import display
pd.set_option('future.no_silent_downcasting', True)

### Load each coded folder set

In [ ]:
### The exact files and file ordering from the HPC workspace Febuary 2025: 
with open("coder_lists_order_snapshot_13Feb2025_local.json", "r", encoding="utf-8") as f:
    snapshot = json.load(f)

## Build one cleaned dataframe per coder, while preserving the original file and row ordering.
dfs = list()
# These are the columns that contain the coder labels we want to clean and later suffix by coder number.
score_cols = ['anger', 'fear', 'superiority']
# Loop through the three coder file lists in the fixed order from the snapshot.
for i, coder in enumerate(["code1", "code2", "code3"], start = 1):
    
    # Pull the files for the current coder in the exact stored order for reproducibility.
    files = snapshot[coder]

    # Collect the raw file-level dataframes before stacking them for this coder.
    coder_dfs = list()
    # Read each Excel file exactly in order so later drop_duplicates keeps the last observed row.
    for file in files:
        temp = pd.read_excel(file)

        # Some files store the TikTok id as a numeric video_id, others only as a full TikTok url.
        id_col = 'video_id' if 'video_id' in temp.columns else 'url'
        # Some files use party_cat while others may already use party.
        party_col = 'party_cat' if 'party_cat' in temp.columns else 'party'

        # Keep only the columns needed downstream and standardize the id/party column names.
        temp = temp[[id_col, party_col, 'user_name', 'anger', 'fear', 'superiority']].rename(
            columns = {id_col: 'tiktok_id', party_col: 'party'}
        )
        
        # If the id came from the url column, extract the numeric TikTok id from the end of the url.
        if id_col == 'url':
            temp.tiktok_id = temp.tiktok_id.apply(lambda x: int(str(x).split("/")[-1]))

        # Store all ids as nullable integers so the merge key has one consistent dtype.
        # Keep the source file name because it is needed later for tracing rows back to their input file.
        temp['file'] = os.path.basename(file)
        # Append the cleaned file-level dataframe without changing row order.
        coder_dfs.append(temp)

    # Stack all files for this coder into one dataframe, preserving file order and within-file row order.
    coder_df = pd.concat(coder_dfs, ignore_index = True)
    # Standardize non-substantive placeholders used in the coder sheets.
    coder_df[score_cols] = coder_df[score_cols].replace(['-', '.'], np.nan)
    # Recode 9 / '9' to 0 to match the original coding logic.
    coder_df[score_cols] = coder_df[score_cols].replace([9, '9'], 0)
    # Convert the score columns to nullable integers after cleaning.
    coder_df[score_cols] = coder_df[score_cols].apply(pd.to_numeric, errors = 'coerce').astype('Int64')
    # Suffix each score column with the coder number so the three coder dataframes can be merged together.
    coder_df.rename(columns={col: col + f'_{i}' for col in score_cols}, inplace=True)
    # Original random order-dependent step: keep the last occurrence of each tiktok_id after concatenation.
    coder_df.drop_duplicates(subset = "tiktok_id", keep = "last", inplace = True)
    
    # Store the cleaned coder dataframe in the same coder order as the loop.
    dfs.append(coder_df)


# Start from coder 1 and add coder 2 and coder 3 scores onto the shared tiktok_id key.
df = dfs[0]
# Only merge in the suffixed score columns from the remaining coder dataframes.
for i, coder_df in enumerate(dfs[1:], start = 2):
    df = df.merge(
        coder_df[['tiktok_id', f'anger_{i}', f'fear_{i}', f'superiority_{i}']],
        on = 'tiktok_id'
    )

# Reorder the final columns so the merged dataframe matches the layout used downstream.
df = df[['file', 'tiktok_id', 'party', 'user_name', 'anger_1', 'fear_1', 'superiority_1',
       'anger_2', 'fear_2', 'superiority_2', 'anger_3', 'fear_3',
       'superiority_3']]
print(df.shape)

cols_to_convert = [col for col in df.columns.tolist() if col not in ['file', 'tiktok_id', 'party', 'user_name']]
df.dropna(subset=cols_to_convert, how='all', inplace = True)
df.reset_index(drop = True, inplace = True)

df.to_parquet("data/local_coded.parquet")
print(df.shape)
df.head(3)


(1330, 13)
(1327, 13)


,file,tiktok_id,party,user_name,anger_1,fear_1,superiority_1,anger_2,fear_2,superiority_2,anger_3,fear_3,superiority_3
0,ra_6_political.xlsx,7429725237949795616,Linke,die.linke.hamburg,0,0,0,0,0,0,0,0,0
1,ra_6_political.xlsx,7429402389330939168,Linke,die.linke.hamburg,0,0,0,0,0,0,0,0,0
2,ra_6_political.xlsx,7451969045944978710,AfD,peterfelser_mdb,0,0,0,0,0,0,<NA>,<NA>,<NA>


### SM Table C.1

In [ ]:
# Reuse the coder rounds and emotion labels across the agreement checks below.
emotion_categories = ['anger', 'fear', 'superiority']
coder_rounds = ['1', '2', '3']
# Keep a single list of the score columns so all later filtering is explicit.
emotion_cols = [f'{emotion}_{coder}' for coder in coder_rounds for emotion in emotion_categories]
# Use the already cleaned dataframe for the agreement statistics below.

# Compute Cohen's Kappa for a given pair of coder columns after removing missing values.
def get_pairwise_kappa(data, col1, col2):
    comp = data[[col1, col2]].dropna()
    if comp.empty:
        return np.nan, 0
    return cohen_kappa_score(comp[col1], comp[col2]), comp.shape[0]

# Collect one summary row per emotional label.
kappa_rows = list()
for emotion in emotion_categories:
    row = {'label': emotion}

    # Compute pairwise kappas for the three coder combinations.
    for coder_1, coder_2 in [('1', '2'), ('1', '3'), ('2', '3')]:
        kappa, n = get_pairwise_kappa(df, f'{emotion}_{coder_1}', f'{emotion}_{coder_2}')
        row[f'kappa_{coder_1}{coder_2}'] = round(kappa, 3) if not np.isnan(kappa) else np.nan
        row[f'n_{coder_1}{coder_2}'] = n

    # Store the mean kappa across the available coder pairs for this label.
    valid_kappas = [row['kappa_12'], row['kappa_13'], row['kappa_23']]
    valid_kappas = [k for k in valid_kappas if not np.isnan(k)]
    row['mean_kappa'] = round(np.mean(valid_kappas), 3) if valid_kappas else np.nan
    kappa_rows.append(row)

# Build a binary any-emotion label for each coder: 1 if any emotion is present, 0 if all are absent.
any_emotion_df = pd.DataFrame({'tiktok_id': df.tiktok_id})
for coder in coder_rounds:
    temp = df[[f'{emotion}_{coder}' for emotion in emotion_categories]]
    any_emotion_df[f'any_emotion_{coder}'] = np.where(
        temp.eq(1).any(axis = 1),
        1,
        np.where(temp.eq(0).all(axis = 1), 0, np.nan)
    )

# Compute the same pairwise kappas for any emotional label versus none.
row = {'label': 'any_emotion_vs_none'}
for coder_1, coder_2 in [('1', '2'), ('1', '3'), ('2', '3')]:
    kappa, n = get_pairwise_kappa(any_emotion_df, f'any_emotion_{coder_1}', f'any_emotion_{coder_2}')
    row[f'kappa_{coder_1}{coder_2}'] = round(kappa, 3) if not np.isnan(kappa) else np.nan
    row[f'n_{coder_1}{coder_2}'] = n

# Add the binary any-emotion result to the same summary table.
valid_kappas = [row['kappa_12'], row['kappa_13'], row['kappa_23']]
valid_kappas = [k for k in valid_kappas if not np.isnan(k)]
row['mean_kappa'] = round(np.mean(valid_kappas), 3) if valid_kappas else np.nan
kappa_rows.append(row)

# Display all kappa results in one table.
kappa_table = pd.DataFrame(kappa_rows)
kappa_table


,label,kappa_12,n_12,kappa_13,n_13,kappa_23,n_23,mean_kappa
0,anger,0.489,1325,0.473,791,0.514,791,0.492
1,fear,0.511,1325,0.422,791,0.426,791,0.453
2,superiority,0.394,1325,0.411,791,0.308,791,0.371
3,any_emotion_vs_none,0.550,1327,0.299,1327,0.362,1327,0.404


In [ ]:
# Print the mean kappa results as a LaTeX table for the paper.
mean_kappas = kappa_table.set_index('label')['mean_kappa']

latex_table = f"""\\begin{{table}}[ht]
    \\centering
\\begin{{tabular}}{{|r|r|r|r|}}
\\hline
\\textbf{{Anger}} & \\textbf{{Fear}} & \\textbf{{Superiority}} & \\textbf{{Any emotion}} \\\\
\\hline
{mean_kappas['anger']:.3f} & {mean_kappas['fear']:.3f} & {mean_kappas['superiority']:.3f} & {mean_kappas['any_emotion_vs_none']:.3f} \\\\
\\hline
\\end{{tabular}}
    \\caption{{Average Cohen's Kappa between coder pairings}}
    \\label{{tab:cohens_kappa}}
\\end{{table}}"""

print(latex_table)


\begin{table}[ht]
    \centering
\begin{tabular}{|r|r|r|r|}
\hline
\textbf{Anger} & \textbf{Fear} & \textbf{Superiority} & \textbf{Any emotion} \\
\hline
0.492 & 0.453 & 0.371 & 0.404 \\
\hline
\end{tabular}
    \caption{Average Cohen's Kappa between coder pairings}
    \label{tab:cohens_kappa}
\end{table}


### Create final set - one category per emotion across coders

In [ ]:
# Start a consensus dataframe that will store the agreed label per emotion and tiktok_id.
newdf = pd.DataFrame(columns = ['tiktok_id', 'anger', 'fear', 'superiority'])

# Apply the same rule from test.ipynb to build a consensus label for each emotion.
for emotion in emotion_categories:
    emdf = pd.concat([df[f'{emotion}_{x}'] for x in coder_rounds], axis = 1)

    # Use the mode when all three coders are present.
    for index, row in emdf.iterrows():
        na_status = any(row.isna())
        if not na_status:
            newdf.loc[index, 'tiktok_id'] = df.loc[index, 'tiktok_id']
            newdf.loc[index, emotion] = row.mode()[0]

        # If one coder is missing, only keep the row when the available coders agree.
        if na_status:
            r = row.dropna()
            if r.nunique() == 1:
                newdf.loc[index, 'tiktok_id'] = df.loc[index, 'tiktok_id']
                newdf.loc[index, emotion] = r.unique()[0]

# Flag rows that end up with more than one emotional category and remove them.
newdf['multicat'] = np.where(newdf.iloc[:, 1:].sum(axis = 1) > 1, 1, 0)
newdf.query("multicat == 0", inplace = True)
newdf.drop(columns = ['multicat'], inplace = True)

# Collapse the one-hot emotion columns into a single category label.
def assign_emotion(row):
    emotions = ['anger', 'fear', 'superiority']
    
    # Find which emotion columns are equal to 1.
    active_emotions = [emotion for emotion in emotions if row[emotion] == 1]

    # If there's at least one emotion with a value of 1, assign it.
    if active_emotions:
        return active_emotions[0]
    else:
        return 'neutral'

# Add the final consensus category to the dataframe.
newdf['category'] = newdf.apply(assign_emotion, axis = 1)

# Bring back party and username metadata for the category summaries.
newdf = newdf.merge(df[['tiktok_id', 'party', 'user_name']], on = 'tiktok_id', how = 'left')
newdf = newdf[['tiktok_id', 'party', 'user_name', 'category']]

# Build the overall category distribution table.
category_dist = newdf.category.value_counts().rename_axis('category').reset_index(name = 'count')
# Build the category distribution within each party.
party_category_dist = newdf.groupby('party').category.value_counts().rename('count').reset_index()


newdf.to_parquet("data/local_political.parquet")

# Display both category distribution tables.
display(category_dist)
display(party_category_dist)


,category,count
0,neutral,980
1,superiority,123
2,anger,90
3,fear,51


,party,category,count
0,AfD,neutral,108
1,AfD,superiority,39
2,AfD,anger,27
3,AfD,fear,14
4,BSW,neutral,23
5,BSW,superiority,4
6,BSW,fear,2
7,BSW,anger,1
8,CDU_CSU,neutral,190
9,CDU_CSU,superiority,13


**Interpretation of the rules**

Each coding section is treated as one separate coder. The final category is therefore intended to represent the average/final coder consensus across the three coder sections.

The consensus is constructed emotion by emotion, not directly at the final category level. For each of the three binary emotions (`anger`, `fear`, `superiority`):

- If all three coder sections are present, the majority decision is used.
- If one coder section is missing, the emotion is only kept when the two available coder sections agree.
- If one coder section is missing and the two available coder sections disagree, that emotion is left unresolved and is not written into the consensus dataframe for that row.

After this emotion-level consensus step, the code removes rows where the final consensus would imply more than one emotional category at once. This means the `multicat` removal applies to the final consensus verdict only, not to the raw coder-round labels. A video can therefore still have been marked with multiple emotions within individual coder sections; what is removed are cases where the final aggregated verdict across coder sections still contains more than one active emotion.

I agree with that interpretation.

The final category assignment then works as follows:

- If the final consensus has `anger == 1`, the final category becomes `anger`.
- If the final consensus has `fear == 1`, the final category becomes `fear`.
- If the final consensus has `superiority == 1`, the final category becomes `superiority`.
- If none of the final consensus emotion columns are equal to `1`, the row is classified as `neutral`.

One important consequence is that unresolved disagreement in an emotion with missing coder information does not necessarily remove the whole `tiktok_id`. It only means that this specific emotion is not written into the consensus row. A row drops out entirely only if no usable consensus emotion is written for that row, or if the final consensus row is removed by the `multicat` filter.
